[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Connections and Transactions


## What you will be able to do

Change a database through SQLAlchemy so that a group of changes is saved together or not at all:
committed as you go on a connection from `connect()`, or once, at the end of a `begin()` block. Say
what another connection sees before a commit and after it. Undo one part of a transaction with a
savepoint, so that a batch carries on past a change that fails. Recognize the rows that were never
saved because nothing committed them, and the errors from a transaction used after it ended.


## The idea

### The problem

Registration for Spring 2026 is open, and every evening the registrar's office sends the database the
day's section changes: a student moving from Composition to World History, another from Calculus I to
Calculus II. A change is two statements, a `DELETE` of the enrollment in the old section and an
`INSERT` of one in the new. If the second fails, because the new section does not exist or the
student is already in it, the first has to be undone too, or the student leaves the evening enrolled
in neither. And one bad change in a batch of forty must not undo the thirty-nine good ones.

There is a quieter failure too. Every `with engine.connect()` block in the **Engines and URLs**
notebook ended with `ROLLBACK` in the log. A program that inserts rows on such a connection and never
commits has changed nothing, and nothing says so: the rows are there when the program reads them back
in the same block, and gone the next time anybody looks.

### What a transaction is

> A **transaction** is a group of statements that the database saves together, with a **commit**,
> or undoes together, with a **rollback**. A SQLAlchemy **`Connection`** runs its statements inside
> one transaction at a time, which it begins by itself when the first statement runs, called
> **autobegin**. **`engine.connect()`** hands over a connection whose changes are committed as you
> go, with **`conn.commit()`**, and whose block rolls back whatever is left when it ends.
> **`engine.begin()`** hands over a connection whose whole block is one transaction, committed when
> the block ends, or rolled back if an exception leaves it. A **savepoint**, made with
> **`conn.begin_nested()`**, marks a point inside a transaction that the transaction can roll back to
> without undoing what came before it.

### Why it works that way

- **Nothing is saved until something commits.** A change made in a transaction exists only inside
  it, and the end of a `connect()` block rolls back anything that was not committed, so a forgotten
  `commit()` loses the change without an error.
- **`begin()` ties the transaction to the block.** An exception that leaves the block rolls back
  every statement in it, whether the database raised it or the program did, so a change made of two
  statements is never half made.
- **Other connections see a change only after its commit.** Until then they read the data as it was,
  so nobody else ever sees the moment between the `DELETE` and the `INSERT`.
- **A savepoint is a transaction inside a transaction.** Rolling back to it undoes only what came
  after it, so a batch can put every change in a savepoint of its own and let a change that fails
  take only itself back.
- **SQLite's driver has to leave transactions to SQLAlchemy.** By default, `sqlite3` begins a
  transaction only before a statement that changes rows, so a savepoint that comes first starts a
  transaction of its own and is committed the moment it is released. This notebook's engine passes
  `autocommit=False`, which keeps a transaction open at all times, and the last of the Common errors
  shows what goes wrong without it.
- **A transaction that ended stays ended.** Committing it twice, using it after its rollback, or
  running a statement on a connection whose block has closed is an error, rather than a guess at what
  the program meant.

### Where this shows up

The ORM's `Session`, the subject of **The Session** notebook, runs its work in these transactions:
`session.commit()` commits the transaction of the connection underneath it. A web service written
with FastAPI or Flask usually gives every request a transaction of its own, committed when the
request succeeds and rolled back when it fails. The **Testing a Data Layer** notebook runs every
test inside a transaction it rolls back, so that no test sees what another left behind. The
**Transactions** notebook of the **sqlite3, Deep Dive** guide covered the same ideas one level down,
with `BEGIN`, `COMMIT` and `SAVEPOINT` written by hand, and its **autocommit and isolation_level**
notebook the driver's modes.

### What this notebook covers

- Commit as you go, with `engine.connect()` and `commit()`
- Begin once, with `engine.begin()`, and the rollback an exception brings
- What another connection sees before and after a commit
- Savepoints with `begin_nested()`, in a batch of section moves where one move fails
- `connect()`, `begin()` or `begin_nested()`: which to use when
- The evening batch, finished
- Six errors, from the enrollment nobody committed to the savepoint a rollback missed

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///college.db")
with engine.begin() as conn:                     # one transaction, committed when the block ends
    conn.execute(text("CREATE TABLE enrollments (student TEXT, course TEXT)"))
    conn.execute(text("INSERT INTO enrollments VALUES ('Ana Reyes', 'STA-200')"))

with engine.connect() as conn:                   # a change here is kept only if it is committed
    conn.execute(text("INSERT INTO enrollments VALUES ('Ben Okafor', 'STA-200')"))
    print("inside the block:", conn.execute(text("SELECT COUNT(*) FROM enrollments")).scalar_one())

with engine.connect() as conn:
    print("the next block:  ", conn.execute(text("SELECT COUNT(*) FROM enrollments")).scalar_one())
```

```
inside the block: 2
the next block:   1
```

The first block committed when it ended. The second inserted a row, counted it, and ended without
committing, so its row was rolled back, and the next block found only the first.


## Setup

Eight imports, the college's database, and the engine helper.

- `sqlalchemy` is the library itself, and the cell prints its version
- `create_engine`, `event` and `text`, from `sqlalchemy`, make the engine, switch on its foreign
  keys, and run SQL
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `IntegrityError`, from `sqlalchemy.exc`, is the error a refused change raises
- `logging` carries the SQL an engine logs to `PrintStatements`
- `sqlite3` builds the database, `Path` names the files, and `shutil` removes the scratch folder at
  the start and at the end

The database, `scratch/college.db`, has grown three tables since the **Why SQLAlchemy** notebook. All
of its rows come from the lists and loops in this cell:

- `students`, 25 of them, and `courses`, 10, as before
- `terms`: Fall 2024, Spring 2025, Fall 2025 and Spring 2026, which is under way
- `sections`: one section of every course in every term, 40 in all, so the section of course `c` in
  term `t` has the id `(t - 1) * 10 + c`, and Spring 2026's run from 31 to 40
- `enrollments`: a row for every student in every section, with a `status` and a `grade`. Every
  student takes three courses a term from the term they started, completed with a grade in the
  first three terms and enrolled, with no grade yet, in Spring 2026: 228 rows

`PrintStatements` is the handler the **Engines and URLs** notebook wrote, and `college_engine` is its
engine helper with one change: it passes `autocommit=False` to `sqlite3`, and switches `autocommit`
on for a moment around the `PRAGMA`, which does nothing inside a transaction. The last of the Common
errors shows why the change is there.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import shutil
import sqlite3
from pathlib import Path

import sqlalchemy
from sqlalchemy import create_engine, event, text
from sqlalchemy.exc import IntegrityError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL);
    CREATE TABLE sections (id INTEGER PRIMARY KEY, course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id), capacity INTEGER NOT NULL);
    CREATE TABLE enrollments (student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL, grade TEXT,
                              PRIMARY KEY (student_id, section_id));
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.executemany("INSERT INTO terms (name, starts_on) VALUES (?, ?)", TERMS)
build.executemany("INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)", SECTIONS)
build.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)", ENROLLMENTS)
build.commit()
build.close()

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", len(STUDENTS), "students,", len(COURSES), "courses,",
      len(TERMS), "terms,", len(SECTIONS), "sections,", len(ENROLLMENTS), "enrollments")


sqlalchemy 2.0.54 | scratch/college.db | 25 students, 10 courses, 4 terms, 40 sections, 228 enrollments


## Worked examples

### The statements this notebook runs

Every change in this notebook is made of two statements written with `text()`, `ADD` and `DROP`, and
`spring_courses` reads back the courses a student takes in Spring 2026. `SPRING_2026` turns a course
code into the id of its Spring 2026 section, so that a change can name courses rather than numbers:


In [2]:
ADD = text("INSERT INTO enrollments (student_id, section_id, status) VALUES (:student, :new, 'enrolled')")
DROP = text("DELETE FROM enrollments WHERE student_id = :student AND section_id = :old")
SPRING_COURSES = text("""
    SELECT courses.code FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    WHERE enrollments.student_id = :student AND sections.term_id = 4
    ORDER BY courses.code
""")
SPRING_2026 = {code: 30 + number for number, (code, *_) in enumerate(COURSES, start=1)}   # course code -> section


def spring_courses(conn, student):
    """The codes of the courses a student takes in Spring 2026."""
    return conn.execute(SPRING_COURSES, {"student": student}).scalars().all()


engine = college_engine(DATABASE)

with engine.connect() as conn:
    for student in (1, 2):
        print(NAMES[student - 1], spring_courses(conn, student))
print("Statistics is section", SPRING_2026["STA-200"])


Ana Reyes ['BIO-101', 'CSC-101', 'HIS-110']
Ben Okafor ['CHE-110', 'CSC-201', 'PSY-101']
Statistics is section 40


`engine` is the only engine most of this notebook uses, made once with `college_engine`, and every
block below borrows a connection from it.

### Commit as you go

A connection from `engine.connect()` begins a transaction with its first statement, and `commit()`
saves what the transaction did. Ana Reyes adds Statistics, and her change is committed. Ben Okafor
adds it in the same block, and the block ends before anything commits his:


In [3]:
engine.echo = True
with engine.connect() as conn:
    conn.execute(ADD, {"student": 1, "new": SPRING_2026["STA-200"]})        # Ana Reyes adds Statistics
    conn.commit()
    conn.execute(ADD, {"student": 2, "new": SPRING_2026["STA-200"]})        # Ben Okafor too, never committed
engine.echo = False

with engine.connect() as conn:
    print("Ana Reyes: ", spring_courses(conn, 1))
    print("Ben Okafor:", spring_courses(conn, 2))


    BEGIN (implicit)
    INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')
    values: (1, 40)
    COMMIT
    BEGIN (implicit)
    INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')
    values: (2, 40)
    ROLLBACK
Ana Reyes:  ['BIO-101', 'CSC-101', 'HIS-110', 'STA-200']
Ben Okafor: ['CHE-110', 'CSC-201', 'PSY-101']


The log shows two transactions on one connection. The first ran Ana's `INSERT` and committed, and the
statement after the commit began a second, which ran Ben's `INSERT` and was rolled back when the
block ended. `conn.commit()` can be called as often as a job needs, which suits one that saves its
work in stages, such as a load committed every thousand rows.

### Begin once

A section move is two statements that must succeed or fail together, which is what `engine.begin()`
is for: the block is one transaction, committed when it ends, or rolled back if an exception leaves
it. Chloe Martin's move succeeds. Daniel Kim's names section 41, which does not exist, and the
helper's foreign keys refuse it:


In [4]:
def move(engine, student, old, new):
    """Move a student from one section to another: two statements, one transaction."""
    with engine.begin() as conn:
        conn.execute(DROP, {"student": student, "old": old})
        conn.execute(ADD, {"student": student, "new": new})


move(engine, 3, SPRING_2026["ENG-105"], SPRING_2026["HIS-110"])     # Chloe Martin: Composition to World History

engine.echo = True
try:
    move(engine, 4, SPRING_2026["MAT-121"], 41)                      # Daniel Kim: Calculus II to no such section
except IntegrityError as error:
    print("not moved:", error.orig)
engine.echo = False

with engine.connect() as conn:
    print("Chloe Martin:", spring_courses(conn, 3))
    print("Daniel Kim:  ", spring_courses(conn, 4))


    BEGIN (implicit)
    DELETE FROM enrollments WHERE student_id = ? AND section_id = ?
    values: (4, 34)
    INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')
    values: (4, 41)
    ROLLBACK
not moved: FOREIGN KEY constraint failed
Chloe Martin: ['HIS-110', 'MAT-120', 'STA-200']
Daniel Kim:   ['BIO-101', 'HIS-110', 'MAT-121']


Daniel's `DELETE` ran, the `INSERT` failed, and the exception left the block, so `begin()` rolled
back the `DELETE` too: Daniel still has Calculus II, and the `ROLLBACK` in the log is the moment
that happened. `error.orig` is the driver's own error inside SQLAlchemy's. Chloe's move, in a
transaction of its own, committed.

### What another connection sees

A change is invisible to every other connection until it is committed. A clerk adds Calculus I for
Elena Petrova, and an advisor, on a second connection, looks before the clerk commits and again
after:


In [5]:
with engine.connect() as clerk:
    clerk.execute(ADD, {"student": 5, "new": SPRING_2026["MAT-120"]})      # Elena Petrova adds Calculus I
    print("the clerk, before the commit:   ", spring_courses(clerk, 5))
    with engine.connect() as advisor:
        print("the advisor, before the commit: ", spring_courses(advisor, 5))
    clerk.commit()

with engine.connect() as advisor:
    print("the advisor, after the commit:  ", spring_courses(advisor, 5))


the clerk, before the commit:    ['CHE-110', 'CSC-101', 'MAT-120', 'PSY-101']
the advisor, before the commit:  ['CHE-110', 'CSC-101', 'PSY-101']
the advisor, after the commit:   ['CHE-110', 'CSC-101', 'MAT-120', 'PSY-101']


The clerk's own connection sees the new enrollment at once, and the advisor's does not see it until
the commit. SQLite also makes a commit wait for every other connection that is part way through a
transaction that has read from the database, which is why the advisor's first block ends before the
clerk commits. The **Concurrency and WAL** notebook of the **sqlite3, Deep Dive** guide covers those
locks, and a server such as PostgreSQL lets a reader and a writer go on at the same time.

### Savepoints, and a batch where one change fails

A batch of moves needs more than one transaction can give. With `begin()` around the whole batch,
one bad move rolls back all of them. Without a transaction for every move, a move whose `INSERT`
fails keeps its `DELETE`. This trial run shows the second, in a block that never commits, so that
nothing it does is saved:


In [6]:
MOVES = [
    {"student": 6, "old": SPRING_2026["CSC-201"], "new": SPRING_2026["CSC-101"]},   # Felix Wagner
    {"student": 7, "old": SPRING_2026["ENG-105"], "new": 41},                       # Grace Lin, to no such section
    {"student": 8, "old": SPRING_2026["HIS-110"], "new": SPRING_2026["PSY-101"]},   # Hassan Ali
]

with engine.connect() as conn:                   # a trial run: nothing in this block is committed
    for change in MOVES:
        try:
            conn.execute(DROP, change)
            conn.execute(ADD, change)
        except IntegrityError as error:
            print("refused:", change, "|", error.orig)
    print("Grace Lin, during the trial run:", spring_courses(conn, 7))

with engine.connect() as conn:
    print("Grace Lin, after its rollback:  ", spring_courses(conn, 7))


refused: {'student': 7, 'old': 37, 'new': 41} | FOREIGN KEY constraint failed
Grace Lin, during the trial run: ['BIO-101', 'MAT-121']
Grace Lin, after its rollback:   ['BIO-101', 'ENG-105', 'MAT-121']


The `try` caught the refused `INSERT`, and the batch went on, but Grace Lin's `DELETE` had already
run, and during the trial she was enrolled in neither Composition nor anything to replace it. On
SQLite a failed statement undoes only itself, which is why the `DELETE` stayed. PostgreSQL goes
further, and refuses every later statement in a transaction until it is rolled back, so there the
rest of the batch would have failed as well.

A savepoint gives every move a transaction of its own inside the batch's. `conn.begin_nested()` sets
one, and used with `with`, it releases the savepoint when the block succeeds and rolls back to it
when an exception leaves the block:


In [7]:
engine.echo = True
with engine.begin() as conn:
    for change in MOVES:
        try:
            with conn.begin_nested():
                conn.execute(DROP, change)
                conn.execute(ADD, change)
        except IntegrityError as error:
            print("refused:", change, "|", error.orig)
engine.echo = False

with engine.connect() as conn:
    for student in (6, 7, 8):
        print(f"{NAMES[student - 1]:<12}", spring_courses(conn, student))


    BEGIN (implicit)
    SAVEPOINT sa_savepoint_1
    DELETE FROM enrollments WHERE student_id = ? AND section_id = ?
    values: (6, 36)
    INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')
    values: (6, 35)
    RELEASE SAVEPOINT sa_savepoint_1
    SAVEPOINT sa_savepoint_2
    DELETE FROM enrollments WHERE student_id = ? AND section_id = ?
    values: (7, 37)
    INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')
    values: (7, 41)
    ROLLBACK TO SAVEPOINT sa_savepoint_2
refused: {'student': 7, 'old': 37, 'new': 41} | FOREIGN KEY constraint failed
    SAVEPOINT sa_savepoint_3
    DELETE FROM enrollments WHERE student_id = ? AND section_id = ?
    values: (8, 38)
    INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')
    values: (8, 39)
    RELEASE SAVEPOINT sa_savepoint_3
    COMMIT
Felix Wagner ['CSC-101', 'MAT-120', 'STA-200']
Grace Lin    ['BIO-101', 'ENG-105', 'MAT-121']
Hassan Al

Every move ran between a `SAVEPOINT` and a `RELEASE SAVEPOINT`, except Grace Lin's, whose failed
`INSERT` sent `ROLLBACK TO SAVEPOINT sa_savepoint_2` and took her `DELETE` back with it. The batch
went on, and the `COMMIT` at the end saved the two moves that worked: Felix Wagner and Hassan Ali
moved, and Grace Lin kept Composition. SQLAlchemy numbers its savepoints for itself, and the `try`
sits outside the `with` so that the savepoint has rolled back before the program decides what to do
next.

### connect(), begin() or begin_nested()

| Use | When | Why |
|---|---|---|
| `engine.begin()` | a change that must succeed or fail as one, such as a move or a transfer | the block commits when it ends and rolls back on any exception, so there is no commit to forget |
| `engine.connect()` and `commit()` | a long job that saves in stages, such as a load committed every thousand rows | the program decides when to commit, and the block rolls back whatever is left |
| `conn.begin_nested()` | one part of a transaction that may fail on its own, such as one request in a batch | a failure rolls back to the savepoint and leaves the rest of the transaction alone |
| `engine.connect()`, with no commit | reading | the rollback when the block ends costs a read nothing |

The default for anything that writes is `engine.begin()`. A savepoint goes inside it, around every
part that is allowed to fail, and `connect()` with `commit()` is for a job that has a reason to
commit more than once.

### The evening batch, finished

The pieces of this notebook in one function. `apply_moves` runs a batch inside one `begin()` block,
puts every move in a savepoint, and refuses a move for three reasons: the new section does not exist,
the student is already in it, or the student was never in the old one. The last is a rule of the
registrar's rather than the database's, and `rowcount`, the number of rows the `DELETE` removed, is
how the function finds out:


In [8]:
class NotEnrolled(Exception):
    """The student in a move is not enrolled in the section they are moving out of."""


def apply_moves(engine, moves):
    """Apply a batch of section moves, each one all or nothing, and return the moves refused, with the reason."""
    refused = []
    with engine.begin() as conn:
        for change in moves:
            try:
                with conn.begin_nested():
                    if conn.execute(DROP, change).rowcount == 0:
                        raise NotEnrolled("not enrolled in the old section")
                    conn.execute(ADD, change)
            except IntegrityError as error:
                refused.append((change, str(error.orig)))
            except NotEnrolled as error:
                refused.append((change, str(error)))
    return refused



EVENING = [
    {"student": 9, "old": SPRING_2026["PSY-101"], "new": SPRING_2026["STA-200"]},    # Isabel Costa
    {"student": 10, "old": SPRING_2026["ENG-105"], "new": SPRING_2026["STA-200"]},   # Jonas Berg, already in Statistics
    {"student": 11, "old": SPRING_2026["MAT-120"], "new": SPRING_2026["MAT-121"]},   # Keiko Tanaka, not in Calculus I
    {"student": 12, "old": SPRING_2026["CHE-110"], "new": 41},                       # Liam Murphy, to no such section
    {"student": 13, "old": SPRING_2026["ENG-105"], "new": SPRING_2026["HIS-110"]},   # Maya Patel
]

for change, reason in apply_moves(engine, EVENING):
    print("refused:", NAMES[change["student"] - 1], "|", reason)

with engine.connect() as conn:
    for student in range(9, 14):
        print(f"{NAMES[student - 1]:<13}", spring_courses(conn, student))


refused: Jonas Berg | UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
refused: Keiko Tanaka | not enrolled in the old section
refused: Liam Murphy | FOREIGN KEY constraint failed
Isabel Costa  ['CSC-201', 'MAT-120', 'STA-200']
Jonas Berg    ['ENG-105', 'MAT-121', 'STA-200']
Keiko Tanaka  ['BIO-101', 'CSC-101', 'HIS-110']
Liam Murphy   ['CHE-110', 'CSC-201', 'PSY-101']
Maya Patel    ['HIS-110', 'MAT-120', 'STA-200']


Five moves, three refused, each for a different reason, and each refusal left its student where they
were: Jonas Berg is still in Composition, Keiko Tanaka's courses never changed, and Liam Murphy kept
General Chemistry. Isabel Costa and Maya Patel moved, and the one `COMMIT` saved both. Raising
`NotEnrolled` inside the savepoint's block rolls the savepoint back the same way a refused `INSERT`
does, although the `DELETE` removed nothing, so a rule of the program's own gets the same treatment
as a rule of the database's.

### Where each part came from

| In `apply_moves` | What it relies on | The section that showed it |
|---|---|---|
| `with engine.begin() as conn:` around the batch | one transaction, committed at the end or rolled back on an exception | Begin once |
| `with conn.begin_nested():` around every move | a savepoint that a failure rolls back to, leaving the rest | Savepoints, and a batch where one change fails |
| `try` outside the savepoint's block | the savepoint rolled back before the program handles the error | Savepoints, and a batch where one change fails |
| `rowcount == 0` raising `NotEnrolled` | an exception of the program's own, rolled back like the database's | The evening batch, finished |
| `college_engine`'s foreign keys | a section that does not exist refused by the database | Begin once |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/03-connections-and-transactions-solutions.ipynb).

**1.** Enroll Noah Andersen, student 14, in Statistics for Spring 2026 with `engine.connect()` and
`commit()`, and show that a new connection sees it.


In [9]:
# your code here


**2.** On one connection, print `conn.in_transaction()` before the first statement, after it, and
after `conn.commit()`.


In [10]:
# your code here


**3.** In an `engine.begin()` block, enroll Olivia Brandt, student 15, in Statistics, and then raise
a `ValueError` of your own. Catch it outside the block, and show whether the enrollment was saved.


In [11]:
# your code here


**4.** In a block you never commit, drop Pavel Novak, student 16, from every Spring 2026 section with
one `DELETE`. Print how many rows it removed, with `rowcount`, and his courses after the block.


In [12]:
# your code here


**5.** In one `engine.begin()` block, add Quinn Harper, student 17, to Statistics and to section 41,
each in a savepoint of its own, and show which of the two was saved.


In [13]:
# your code here


**6.** Run `apply_moves` on the `EVENING` batch a second time. Which moves does it refuse now, and
why?


In [14]:
# your code here


## Common errors

### No error, and the enrollment gone: a change that nothing committed


In [15]:
def enroll(engine, student, section):
    """Enroll a student and return their courses, which is the mistake: nothing commits the change."""
    with engine.connect() as conn:
        conn.execute(ADD, {"student": student, "new": section})
        return spring_courses(conn, student)


print("what enroll returned:", enroll(engine, 18, SPRING_2026["STA-200"]))        # Rosa Delgado
with engine.connect() as conn:
    print("what was saved:      ", spring_courses(conn, 18))


what enroll returned: ['CHE-110', 'CSC-101', 'HIS-110', 'STA-200']
what was saved:       ['CHE-110', 'CSC-101', 'HIS-110']


`enroll` read Statistics back, since it read inside its own transaction, and returned a list that
looked like success. Then its block ended with no commit, the transaction was rolled back, and Rosa
Delgado was never enrolled. Nothing raised, because nothing went wrong: a connection from
`connect()` keeps a change only when told to. Use `engine.begin()`, which commits when the block
ends:


In [16]:
def enroll(engine, student, section):
    """Enroll a student and return their courses, committed when the block ends."""
    with engine.begin() as conn:
        conn.execute(ADD, {"student": student, "new": section})
        return spring_courses(conn, student)


print("what enroll returned:", enroll(engine, 18, SPRING_2026["STA-200"]))
with engine.connect() as conn:
    print("what was saved:      ", spring_courses(conn, 18))


what enroll returned: ['CHE-110', 'CSC-101', 'HIS-110', 'STA-200']
what was saved:       ['CHE-110', 'CSC-101', 'HIS-110', 'STA-200']


### sqlalchemy.exc.InvalidRequestError: This connection has already initialized a SQLAlchemy Transaction() object via begin() or autobegin; can't call begin() here unless rollback() or commit() is called first.


In [17]:
with engine.connect() as conn:
    before = spring_courses(conn, 19)                                  # Sam Ito's courses, read first
    with conn.begin():
        conn.execute(ADD, {"student": 19, "new": SPRING_2026["STA-200"]})


InvalidRequestError: This connection has already initialized a SQLAlchemy Transaction() object via begin() or autobegin; can't call begin() here unless rollback() or commit() is called first.

Reading Sam Ito's courses was the connection's first statement, and it began a transaction, so by the
time `conn.begin()` asked for one, the connection already had one open. `begin()` on a connection has
to come before its first statement. Open the block with `engine.begin()` instead, and the read and
the change share one transaction:


In [18]:
with engine.begin() as conn:
    before = spring_courses(conn, 19)
    conn.execute(ADD, {"student": 19, "new": SPRING_2026["STA-200"]})
    print(before, "->", spring_courses(conn, 19))


['CSC-201', 'MAT-120', 'PSY-101'] -> ['CSC-201', 'MAT-120', 'PSY-101', 'STA-200']


### sqlalchemy.exc.InvalidRequestError: This transaction is inactive


In [19]:
def enroll_once(engine, student, section):
    """Enroll a student, rolling back if the database refuses, which is the mistake: it commits either way."""
    with engine.connect() as conn:
        transaction = conn.begin()
        try:
            conn.execute(ADD, {"student": student, "new": section})
        except IntegrityError:
            transaction.rollback()
        transaction.commit()


enroll_once(engine, 20, SPRING_2026["STA-200"])                      # Tara Nilsen is already in Statistics


InvalidRequestError: This transaction is inactive

Tara Nilsen was already in Statistics, so the `INSERT` was refused and the `except` rolled the
transaction back. Then the function went on to `commit()` a transaction that had already ended. A
transaction is finished once it has committed or rolled back, and SQLAlchemy raises rather than guess
which of the two the program meant. Let `begin()` decide, and handle the refusal outside it:


In [20]:
def enroll_once(engine, student, section):
    """Enroll a student, and report a refusal instead of raising it."""
    try:
        with engine.begin() as conn:
            conn.execute(ADD, {"student": student, "new": section})
        return "enrolled"
    except IntegrityError as error:
        return f"refused: {error.orig}"


print("Statistics:  ", enroll_once(engine, 20, SPRING_2026["STA-200"]))
print("Biology:     ", enroll_once(engine, 20, SPRING_2026["BIO-101"]))


Statistics:   refused: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
Biology:      enrolled


### sqlalchemy.exc.InvalidRequestError: Can't operate on closed transaction inside context manager.  Please complete the context manager before emitting further commands.


In [21]:
with engine.begin() as conn:
    conn.execute(ADD, {"student": 21, "new": SPRING_2026["STA-200"]})       # Umar Farouk adds Statistics
    conn.commit()
    conn.execute(ADD, {"student": 21, "new": SPRING_2026["PSY-101"]})       # and Introduction to Psychology


InvalidRequestError: Can't operate on closed transaction inside context manager.  Please complete the context manager before emitting further commands.

`engine.begin()` owns its transaction and commits it when the block ends, so the `commit()` in the
middle ended that transaction early, and the block refused to run anything after it. The `commit()`
did its work first, though: Umar Farouk's Statistics was saved, and his Introduction to Psychology
was never tried. Inside `begin()`, leave the commit to the block:


In [22]:
with engine.begin() as conn:
    conn.execute(ADD, {"student": 21, "new": SPRING_2026["PSY-101"]})
    print("Umar Farouk:", spring_courses(conn, 21))


Umar Farouk: ['BIO-101', 'CSC-101', 'HIS-110', 'PSY-101', 'STA-200']


### sqlalchemy.exc.ResourceClosedError: This Connection is closed


In [23]:
with engine.begin() as conn:
    conn.execute(ADD, {"student": 22, "new": SPRING_2026["STA-200"]})       # Vera Kowalski adds Statistics
print("Vera Kowalski:", spring_courses(conn, 22))


ResourceClosedError: This Connection is closed

The name `conn` outlives its block, but the connection behind it went back to the pool when the block
ended, and a closed connection runs nothing. The change itself was committed. Read inside the block,
or borrow a new connection:


In [24]:
with engine.connect() as conn:
    print("Vera Kowalski:", spring_courses(conn, 22))


Vera Kowalski: ['CHE-110', 'CSC-201', 'PSY-101', 'STA-200']


### No error, and a row the rollback should have removed: a savepoint under sqlite3's own transactions


In [25]:
plain = create_engine(f"sqlite:///{DATABASE}")                      # sqlite3's transactions, as create_engine leaves them
with plain.connect() as conn:
    with conn.begin() as batch:
        with conn.begin_nested():
            conn.execute(ADD, {"student": 23, "new": SPRING_2026["BIO-101"]})     # Wes Carter adds Biology
        batch.rollback()                                                         # and the whole batch is undone
with plain.connect() as conn:
    print("Wes Carter:", spring_courses(conn, 23))
plain.dispose()


Wes Carter: ['BIO-101', 'ENG-105', 'MAT-120', 'STA-200']


The batch was rolled back, and Wes Carter's Biology is saved anyway. Left to itself, `sqlite3` begins
a transaction only before a statement that changes rows, so the `SAVEPOINT` was the first thing
SQLite saw, and it started a transaction of its own. Releasing that outermost savepoint committed it,
as the **Transactions** notebook of the **sqlite3, Deep Dive** guide showed, and the `ROLLBACK` found
nothing left to undo. With `autocommit=False`, `sqlite3` keeps a transaction open from the moment it
connects, so the savepoint nests inside it, which is why `college_engine` passes it. The same code,
on the helper's engine, for Yara Haddad:


In [26]:
with engine.connect() as conn:
    with conn.begin() as batch:
        with conn.begin_nested():
            conn.execute(ADD, {"student": 24, "new": SPRING_2026["STA-200"]})     # Yara Haddad adds Statistics
        batch.rollback()
with engine.connect() as conn:
    print("Yara Haddad:", spring_courses(conn, 24))


Yara Haddad: ['BIO-101', 'HIS-110', 'MAT-121']


Nothing was saved, as the rollback meant. SQLAlchemy's own documentation for SQLite recommends
`autocommit=False` on Python 3.12 and later, and describes this savepoint behavior as the reason.

Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [27]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- Nothing is saved until something commits: `commit()` on a connection from `engine.connect()`, or
  the end of an `engine.begin()` block.
- `engine.begin()` makes its block one transaction, rolled back if any exception leaves it, so a
  change made of several statements is never half made.
- Another connection sees a change only after the change is committed.
- `begin_nested()` puts part of a transaction in a savepoint, so a failure takes back only that part,
  and on SQLite that holds only when `sqlite3` runs with `autocommit=False`.
- A transaction that ended stays ended: committing it twice, using it after a rollback, or using its
  connection after the block raises an error instead of guessing.


## What is next

The **Reading Results** notebook reads what a statement hands back: the `Result`, the `Row`s inside
it, `scalars()`, `one()` and `first()`, and the result that has already been read to its end.


---

&#8592; **Previous:** [Engines and URLs](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/02-engines-and-urls.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Reading Results](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/04-reading-results.ipynb) &#8594;
